# Lab 2 - Brain Tumor Segmentation

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import ipywidgets as widgets
from ipywidgets import interact

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## Prepare dataset

In [3]:
class BrainTumorDataset(Dataset):
    def __init__(self, rootpath, transform=None, normalization="minmax"):
        self.transform = transform
        self.normalization = normalization
        self.patients = [
            os.path.join(rootpath, r) for r in os.listdir(rootpath) 
                if os.path.isdir(os.path.join(rootpath, r))
        ]

    def __len__(self):
        return len(self.patients)

    def _loadTensor(self, filepath, normalize=True):
        tensor = torch.tensor(nib.load(filepath).get_fdata(dtype=np.float32)).permute(2, 0, 1)
        
        # Normalizar solo si es una imagen de entrada (voxel)
        if normalize:
            tensor = (tensor - tensor.min()) / (tensor.max() - tensor.min() + 1e-8)  # Min-Max Scaling
        
        return tensor

    def __getitem__(self, idx):
        folder = self.patients[idx]

        modalities = {
            "flair.nii.gz": None,
            "seg.nii.gz": None,
            "t1.nii.gz": None,
            "t1ce.nii.gz": None,
            "t2.nii.gz": None
        }

        for file in os.listdir(folder):
            filepath = os.path.join(folder, file)
            for key in modalities.keys():
                if file.endswith(key):
                    modalities[key] = self._loadTensor(filepath, normalize=(key != "seg.nii.gz"))
                    break

        voxel = torch.stack([
            modalities["flair.nii.gz"], 
            modalities["t1.nii.gz"],
            modalities["t1ce.nii.gz"], 
            modalities["t2.nii.gz"]
        ], dim=0)

        seg = modalities["seg.nii.gz"]
        seg[seg == 4] = 3  # Ajuste de etiquetas (fox method 🦊)
        seg = seg.long()  # Convertir a tipo `long`

        return voxel, seg


## Explore and visualize sample

In [4]:
dataset = BrainTumorDataset("train")

# sample
voxel, seg = dataset[0]

# seg (label):
# 0 = no tumor
# 1, 2, 3 = tipos de tumor

# print(f"seg.shape: {seg.shape}, seg range: {torch.min(seg)},{torch.max(seg)}")

# flair = voxel[0]
# print(f"flair.shape: {flair.shape}, flair range: {torch.min(flair)},{torch.max(flair)}")

# t1 = voxel[1]
# print(f"t1.shape: {t1.shape}, t1 range: {torch.min(t1)},{torch.max(t1)}")

# t1ce = voxel[2]
# print(f"t1ce.shape: {t1ce.shape}, t1ce range: {torch.min(t1ce)},{torch.max(t1ce)}")

# t2 = voxel[3]
# print(f"t2.shape: {t2.shape}, t2 range: {torch.min(t2)},{torch.max(t2)}")


In [5]:
def plot_slice(slice_idx, axis, flair, t1, t1ce, t2, seg):

    if axis == "Coronal":
        flair_slice = flair[:, :, slice_idx]
        t1_slice = t1[:, :, slice_idx]
        t1ce_slice = t1ce[:, :, slice_idx]
        t2_slice = t2[:, :, slice_idx]
        seg_slice = seg[:, :, slice_idx]

    elif axis == "Sagital":
        flair_slice = flair[:, slice_idx, :]
        t1_slice = t1[:, slice_idx, :]
        t1ce_slice = t1ce[:, slice_idx, :]
        t2_slice = t2[:, slice_idx, :]
        seg_slice = seg[:, slice_idx, :]
    else:  # Axial
        flair_slice = flair[slice_idx, :, :]
        t1_slice = t1[slice_idx, :, :]
        t1ce_slice = t1ce[slice_idx, :, :]
        t2_slice = t2[slice_idx, :, :]
        seg_slice = seg[slice_idx, :, :]
        
    fig, axs = plt.subplots(1, 5, figsize=(18, 4))

    axs[0].imshow(flair_slice, cmap="gray")
    axs[0].set_title(f"FLAIR - {axis} Slice {slice_idx}")
    axs[0].axis("off")

    axs[1].imshow(t1_slice, cmap="gray")
    axs[1].set_title(f"T1 - {axis} Slice {slice_idx}")
    axs[1].axis("off")

    axs[2].imshow(t1ce_slice, cmap="gray")
    axs[2].set_title(f"T1CE - {axis} Slice {slice_idx}")
    axs[2].axis("off")

    axs[3].imshow(t2_slice, cmap="gray")
    axs[3].set_title(f"T2 - {axis} Slice {slice_idx}")
    axs[3].axis("off")

    axs[4].imshow(seg_slice, cmap="jet", alpha=0.5)
    axs[4].set_title(f"Segmentation - {axis} Slice {slice_idx}")
    axs[4].axis("off")

    plt.show()

def plot_voxel(voxel, seg):
    flair, t1, t1ce, t2 = voxel
    interact(plot_slice, 
             slice_idx=widgets.IntSlider(min=0, max=flair.shape[0] - 1, step=1, value=flair.shape[0] // 2),
             axis=widgets.RadioButtons(options=["Axial", "Coronal", "Sagital"], value="Axial"),
             flair=widgets.fixed(flair),
             t1=widgets.fixed(t1),
             t1ce=widgets.fixed(t1ce),
             t2=widgets.fixed(t2),
             seg=widgets.fixed(seg))

In [6]:
plot_voxel(voxel, seg)

interactive(children=(IntSlider(value=77, description='slice_idx', max=154), RadioButtons(description='axis', …

## Model

La cantidad de canales de salida $\text{(out\_channels=4)}$ representa las diferentes clases. Cada voxel del volumen de salida contiene un vector de 4 valores, uno por cada canal, representando las probabilidades de pertenecer a cada una de las clases:

    1️⃣ Canal 0 (Clase 0): Tejido sano (sin tumor).
    2️⃣ Canal 1 (Clase 1): Tumor de realce (Enhancing tumor).
    3️⃣ Canal 2 (Clase 2): Edema peritumoral.
    4️⃣ Canal 3 (Clase 3): Núcleo del tumor (Necrosis y tumor no realzante).

In [7]:
class DoubleConv3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv3D, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet3D(nn.Module):
    def __init__(self, in_channels=4, out_channels=4, features=[32, 64, 128, 256]):
        super(UNet3D, self).__init__()
        
        # Encoder (Downsampling path)
        self.encoder = nn.ModuleList()
        self.pools = nn.ModuleList()
        for feature in features:
            self.encoder.append(DoubleConv3D(in_channels, feature))
            self.pools.append(nn.MaxPool3d(kernel_size=2, stride=2))
            in_channels = feature
        
        # Bottleneck
        self.bottleneck = DoubleConv3D(features[-1], features[-1] * 2)
        
        # Decoder (Upsampling path)
        self.upconvs = nn.ModuleList()
        self.decoders = nn.ModuleList()
        for feature in reversed(features):
            self.upconvs.append(nn.ConvTranspose3d(feature * 2, feature, kernel_size=2, stride=2))
            self.decoders.append(DoubleConv3D(feature * 2, feature))
        
        # Final output layer
        self.final_conv = nn.Conv3d(features[0], out_channels, kernel_size=1)
    
    def forward(self, x):
        skip_connections = []
        
        # Encoder
        for i in range(len(self.encoder)):
            x = self.encoder[i](x)
            skip_connections.append(x)
            x = self.pools[i](x)
        
        # Bottleneck
        x = self.bottleneck(x)
        
        # Decoder
        skip_connections = skip_connections[::-1]
        for i in range(len(self.upconvs)):
            x = self.upconvs[i](x)
            skip_connection = skip_connections[i]
            
            # Handle size mismatch due to pooling (padding if needed)
            if x.shape != skip_connection.shape:
                x = F.interpolate(x, size=skip_connection.shape[2:], mode='trilinear', align_corners=False)
            
            x = torch.cat((skip_connection, x), dim=1)
            x = self.decoders[i](x)
        
        return self.final_conv(x)



- Verificamos si hay problemas de dimensión cuando una entrada pasa a través de la red.

In [8]:
# Brain Tumor dataset

voxel, seg = dataset[0] # [flair, t1, t1ce, t2], [segmentation]

# seg (label):
# 0 = no tumor
# 1, 2, 3 = tipos de tumor

# voxel = voxel.unsqueeze(dim=0).to("cuda")
# seg = seg.unsqueeze(dim=0).to("cuda")

print(f"voxel.shape: {voxel.shape}")
print(f"seg.shape: {seg.shape}")

model = UNet3D().to("cuda")

voxel.shape: torch.Size([4, 155, 240, 240])
seg.shape: torch.Size([155, 240, 240])


c:\Users\LENOVO\Desktop\Deep-Learning-course\labs\lab2\venv\Lib\site-packages\torch\nn\modules\module.py:1326: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


In [9]:
# with torch.no_grad():
#     logits = model(voxel)

# print(f"logits.shape: {logits.shape}")

# prediction = torch.argmax(logits, dim=1)
# prediction = prediction.long()

# print(f"prediction.shape: {prediction.shape}")

In [10]:
# criterion = nn.CrossEntropyLoss()

# loss = criterion(logits, seg)
# print(loss)

In [11]:
# random = torch.randn(1, 4, 155, 240, 240).to("cuda")

# loss = criterion(random, seg)
# print(loss)

## Data loader

In [12]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)
print(f"Dataset size: {len(dataloader)}")

Dataset size: 1000


In [13]:
torch.cuda.empty_cache()

total_memory = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)
allocated_memory = torch.cuda.memory_allocated(0) / (1024 ** 2)
cached_memory = torch.cuda.memory_reserved(0) / (1024 ** 2)

print(f"Memoria total: {total_memory:.2f} MB")
print(f"Memoria asignada: {allocated_memory:.2f} MB")
print(f"Memoria en caché: {cached_memory:.2f} MB")

Memoria total: 4095.50 MB
Memoria asignada: 89.01 MB
Memoria en caché: 102.00 MB


In [14]:
for index, (voxels, labels) in enumerate(dataloader):
    voxels, labels = voxels.to("cuda"), labels.to("cuda")
    outputs = model(voxels)
    print(f"{index + 1}, {outputs.shape}")

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## Training

- Pregunta: ¿Cómo se compara el mapa de segmentación (con valores de 0 a 3) y los 4 canales de salida (donde cada canal tiene valores de 0 a 1) usando función de pérdida (loss function)?

In [ ]:
def train_unet3d(model, dataloader, num_epochs=2, batch_size=1, lr=1e-4, device="cuda"):
    
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        
        for voxels, labels in dataloader:
            voxels, labels = voxels.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(voxels)
            
            loss = criterion(outputs, labels.long())
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss/len(dataloader):.4f}")
    
    print("Entrenamiento completado.")

## Validation